In [ ]:
%pip install -q langchain langchain-community chromadb requests beautifulsoup4 python-dotenv pydantic ollama pymupdf

Note: you may need to restart the kernel to use updated packages.


# Legal RAG System - Setup and Testing

This notebook walks through setting up and testing the Legal RAG system.

## Step 1: Verify Dependencies

In [1]:
import sys
import subprocess

print("Python executable:", sys.executable)
print("\nChecking pip list...")
result = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
packages = result.stdout.split('\n')
key_packages = [p for p in packages if any(x in p.lower() for x in ['langchain', 'chromadb', 'requests', 'beautifulsoup', 'dotenv', 'pydantic', 'ollama'])]
for pkg in key_packages:
    print(pkg)

Python executable: d:\Desktop\LawRAG\venv_new\Scripts\python.exe

Checking pip list...
beautifulsoup4                           4.14.3
chromadb                                 1.5.7
langchain                                1.2.15
langchain-core                           1.2.29
ollama                                   0.6.1
pydantic                                 2.13.0
pydantic_core                            2.46.0
pydantic-settings                        2.13.1
python-dotenv                            1.2.2
requests                                 2.33.1
requests-oauthlib                        2.0.0
requests-toolbelt                        1.0.0


In [1]:
# Check if all required packages are installed
import sys

required_packages = {
    'langchain': 'LangChain',
    'chromadb': 'ChromaDB',
    'requests': 'Requests',
    'bs4': 'BeautifulSoup4',
    'dotenv': 'Python-dotenv',
    'pydantic': 'Pydantic',
    'ollama': 'Ollama'
}

print("Checking required packages:\n")
all_installed = True
for module, name in required_packages.items():
    try:
        __import__(module)
        print(f'OK: {name}')
    except ImportError:
        print(f'FAIL: {name}')
        all_installed = False

if all_installed:
    print("\nSUCCESS: All required packages are installed!")
else:
    print("\nWARNING: Some packages are missing")

Checking required packages:

OK: LangChain
OK: ChromaDB
OK: Requests
OK: BeautifulSoup4
OK: Python-dotenv
OK: Pydantic
OK: Ollama

SUCCESS: All required packages are installed!


## Step 2: Check Ollama Connection

In [2]:
import sys
from pathlib import Path
import requests

# Add parent directory to path so we can import config
sys.path.insert(0, str(Path.cwd().parent))

from config.settings import OLLAMA_BASE_URL, OLLAMA_MODEL

print(f"Testing Ollama connection at {OLLAMA_BASE_URL}\n")

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    if response.status_code == 200:
        print(f"OK: Ollama is running at {OLLAMA_BASE_URL}")
        models = response.json().get('models', [])
        if models:
            print(f"\nAvailable models:")
            for model in models:
                print(f"  - {model['name']}")
        else:
            print("\nWARNING: No models found. Run 'ollama pull mistral && ollama pull nomic-embed-text'")
    else:
        print(f"FAIL: Ollama returned status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print(f"FAIL: Cannot connect to Ollama at {OLLAMA_BASE_URL}")
    print("Make sure Ollama is running with: ollama serve")
except Exception as e:
    print(f"FAIL: Error connecting to Ollama: {e}")

Testing Ollama connection at http://localhost:11434

OK: Ollama is running at http://localhost:11434



## Step 3: Process Sample Document

In [3]:
import sys
import json
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from config.settings import RAW_DOCS_DIR, PROCESSED_DIR

# First, ensure PyMuPDF is installed
try:
    import fitz
except ImportError:
    print("Installing PyMuPDF...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import fitz

from src.document_processor import LegalDocumentProcessor

print("Step 1: List available documents\n")
pdf_files = list(RAW_DOCS_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files:")
for i, f in enumerate(pdf_files, 1):
    print(f"  {i}. {f.name}")

if not pdf_files:
    print("\nNo PDF files found in data/raw_documents/")
    print("Please download some legal documents first.")
else:
    print(f"\nStep 2: Processing first PDF file\n")
    
    # Use the first PDF
    pdf_file = pdf_files[0]
    print(f"Processing: {pdf_file.name}\n")
    
    processor = LegalDocumentProcessor(chunk_size=500, chunk_overlap=50)
    
    # Extract text from PDF
    try:
        text = processor.extract_text_from_pdf(str(pdf_file))
        if text:
            print(f"Extracted {len(text)} characters from PDF\n")
            
            # Split by sections
            sections = processor.split_by_sections(text)
            print(f"Found {len(sections)} sections")
            
            # Create chunks with metadata
            chunks = []
            for section_name, section_text in sections:
                section_chunks = processor.chunk_with_overlap(section_text)
                for i, chunk in enumerate(section_chunks):
                    chunks.append({
                        "act": pdf_file.stem,
                        "section": section_name,
                        "chunk_id": f"{pdf_file.stem}_{section_name.replace(' ', '_')}_{i}",
                        "content": chunk,
                        "source": str(pdf_file)
                    })
            
            print(f"Created {len(chunks)} chunks from {len(sections)} sections")
            
            # Save processed chunks
            output_file = PROCESSED_DIR / f"{pdf_file.stem}_chunks.json"
            processor.save_processed_chunks(chunks, str(output_file))
            print(f"\nOK: Saved chunks to {output_file}")
            
            if chunks:
                print(f"\nFirst chunk preview:")
                print(f"  Act: {chunks[0]['act']}")
                print(f"  Section: {chunks[0]['section']}")
                print(f"  Content: {chunks[0]['content'][:100]}...")
        else:
            print("FAIL: Could not extract text from PDF")
    except Exception as e:
        print(f"FAIL: Error processing PDF: {e}")

Step 1: List available documents

Found 3 PDF files:
  1. Federal Constitution (Reprint 2020).pdf
  2. Perlembagaan Persekutuan (Cetakan Semula 2020).pdf
  3. personal-data-protection-act-2010.pdf

Step 2: Processing first PDF file

Processing: Federal Constitution (Reprint 2020).pdf

Extracted 769951 characters from PDF

Found 169 sections
Created 392 chunks from 169 sections
Saved 392 chunks to d:\Desktop\LawRAG\data\processed\Federal Constitution (Reprint 2020)_chunks.json

OK: Saved chunks to d:\Desktop\LawRAG\data\processed\Federal Constitution (Reprint 2020)_chunks.json

First chunk preview:
  Act: Federal Constitution (Reprint 2020)
  Section: Part I
  Content: THE STATES, RELIGION AND LAW OF THE FEDERATION Article 1. Name, States and territories of the Federa...


## Step 4: Initialize RAG and Load Documents

In [5]:
%pip install -q langchain-community

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
import json
from pathlib import Path
import chromadb

# Add parent to path
sys.path.insert(0, str(Path.cwd().parent))

from src.rag_pipeline import LegalRAG
from config.settings import PROCESSED_DIR, CHROMA_DB_PATH

print("Initializing Legal RAG System\n")

# Initialize RAG system
try:
    rag = LegalRAG()
    print("OK: RAG system initialized")
    
    # Delete existing collection to avoid duplicate ID errors
    try:
        rag.client.delete_collection(name="malaysian_law")
        print("Cleared existing collection")
    except:
        pass
    
    # Create fresh collection
    rag.create_collection()
    print("OK: ChromaDB collection created")
    
    # Load processed documents
    json_files = list(PROCESSED_DIR.glob("*_chunks.json"))
    
    if json_files:
        print(f"\nFound {len(json_files)} processed document files")
        
        all_chunks = []
        for json_file in json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                chunks = json.load(f)
                all_chunks.extend(chunks)
                print(f"  Loaded {len(chunks)} chunks from {json_file.name}")
        
        try:
            print(f"\nLoading {len(all_chunks)} chunks into ChromaDB...")
            rag.add_documents(all_chunks)
            print(f"OK: Documents loaded into RAG system")
            print(f"\nRAG system is ready for queries!")
        except Exception as e:
            if "nomic-embed-text" in str(e) and "not found" in str(e):
                print(f"\nWARNING: Embedding model not available")
                print("The Ollama models may not be fully loaded yet.")
                print("Wait a moment and try again.")
            else:
                raise
    else:
        print("WARNING: No processed documents found")
        print("Run the document processing cell first")
        
except Exception as e:
    print(f"FAIL: Error initializing RAG: {e}")
    import traceback
    traceback.print_exc()

Initializing Legal RAG System

OK: RAG system initialized
Collection may already exist: Collection [malaysian_law] already exists
OK: ChromaDB collection created

Found 1 processed document files
  Loaded 392 chunks from Federal Constitution (Reprint 2020)_chunks.json

Loading 392 chunks into ChromaDB...
FAIL: Error initializing RAG: Expected IDs to be unique, found 44 duplicated IDs: Federal Constitution (Reprint 2020)_Part_III_0, Federal Constitution (Reprint 2020)_Section_23_0, Federal Constitution (Reprint 2020)_Section_5_0, Federal Constitution (Reprint 2020)_Section_4_0, Federal Constitution (Reprint 2020)_Section_10_0, ..., Federal Constitution (Reprint 2020)_Part_XIIa_0, Federal Constitution (Reprint 2020)_Section_20_0, Federal Constitution (Reprint 2020)_Section_2_0, Federal Constitution (Reprint 2020)_Section_9_0, Federal Constitution (Reprint 2020)_Section_7_0 in add.


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_9548\3244285420.py", line 37, in <module>
    rag.add_documents(all_chunks)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "d:\Desktop\LawRAG\src\rag_pipeline.py", line 70, in add_documents
    self.collection.add(
    ~~~~~~~~~~~~~~~~~~~^
        ids=ids,
        ^^^^^^^^
    ...<2 lines>...
        metadatas=metadatas
        ^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\Desktop\LawRAG\venv_new\Lib\site-packages\chromadb\api\models\Collection.py", line 107, in add
    add_request = self._validate_and_prepare_add_request(
        ids=ids,
    ...<4 lines>...
        uris=uris,
    )
  File "d:\Desktop\LawRAG\venv_new\Lib\site-packages\chromadb\api\models\CollectionCommon.py", line 103, in wrapper
    return func(self, *args, **kwargs)
  File "d:\Desktop\LawRAG\venv_new\Lib\site-packages\chromadb\api\models\CollectionCommon.py", line 235, in _validate_and_prepare_add_request
    validate_insert_record_set(reco

## Step 5: Test RAG Query

In [ ]:
# Test a query
query = "What is the definition of personal data under Malaysian law?"

print(f"Query: {query}\n")

result = rag.query(query, top_k=3)

print(f"Answer:\n{result['answer']}\n")
print(f"Sources: {result['sources']}")